# ETAPA 3 — Modelo Baseline (Regressão Linear)
### Estrutura completa seguindo o GUIA COMPLETO

## PASSO 1 — Carregar dados

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import os

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

DATA_PATH = '/mnt/data/72e16faf-ad7f-4bb2-8f2e-ce17a9ff2499.csv'
df = pd.read_csv(DATA_PATH)
df.head()


## PASSO 2 — Separar X e y

In [ ]:

TARGET = 'total_views'
X = df.drop(columns=[TARGET], errors='ignore')
y = df[TARGET]

X.shape, y.shape


## PASSO 3 — Dividir dados (60/20/20)

In [ ]:

RANDOM_STATE = 42

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE
)

len(X_train), len(X_val), len(X_test)


## PASSO 4 — Criar e treinar modelo

In [ ]:

modelo = LinearRegression()
modelo.fit(X_train, y_train)

coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coeficiente': modelo.coef_
}).sort_values('Coeficiente', key=abs, ascending=False)

coef_df.head(10)


## PASSO 5 — Fazer previsões

In [ ]:

y_train_pred = modelo.predict(X_train)
y_val_pred = modelo.predict(X_val)


## PASSO 6 — Calcular métricas

In [ ]:

def metricas(nome, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f'--- {nome} ---')
    print('MSE :', mse)
    print('RMSE:', rmse)
    print('MAE :', mae)
    print('R²  :', r2)

    return {'MSE':mse, 'RMSE':rmse, 'MAE':mae, 'R2':r2}

m_train = metricas('Treino', y_train, y_train_pred)
m_val = metricas('Validação', y_val, y_val_pred)


## PASSO 7 — Verificar Overfitting

In [ ]:

diff_r2 = abs(m_train['R2'] - m_val['R2'])
print('Diferença de R²:', diff_r2)

if diff_r2 < 0.10:
    print('✅ Modelo generaliza bem')
else:
    print('❌ Overfitting detectado')


## PASSO 8 — Analisar resíduos

In [ ]:

residuos = y_val - y_val_pred
print('Média dos resíduos:', residuos.mean())
print('Desvio padrão:', residuos.std())
print('Min:', residuos.min(), 'Max:', residuos.max())


## PASSO 9 — Gráfico 1: Predições vs Reais

In [ ]:

plt.figure(figsize=(8,6))
plt.scatter(y_val, y_val_pred, alpha=0.6, edgecolors='k')
mn = min(y_val.min(), y_val_pred.min())
mx = max(y_val.max(), y_val_pred.max())
plt.plot([mn,mx],[mn,mx],'r--')
plt.title('Predições vs Reais')
plt.xlabel('Real')
plt.ylabel('Previsto')
plt.tight_layout()
plt.show()


## PASSO 9 — Gráfico 2: Distribuição dos resíduos

In [ ]:

plt.figure(figsize=(8,6))
plt.hist(residuos, bins=30, edgecolor='black')
plt.axvline(0, color='red', linestyle='--')
plt.title('Distribuição dos Resíduos')
plt.xlabel('Resíduo')
plt.ylabel('Frequência')
plt.tight_layout()
plt.show()


## PASSO 10 — Salvar modelo

In [ ]:

MODEL_PATH = 'baseline_model.pkl'
joblib.dump(modelo, MODEL_PATH)
MODEL_PATH
